In [12]:
import time
from datetime import datetime

import ccxt
import pandas as pd

# 1. Exchange and parameters
exchange = ccxt.binanceus()   # same as ccxt$binanceus()
symbol = "BTC/USDT"
poll_interval = 5  # seconds between API calls
csv_file = "advanced_orderflow_data.csv"

print("Starting advanced data collection. Press Ctrl+C to stop.\n")

# 2. Main loop
# 15000 iterations at 5 seconds ≈ 20.8 hours
for i in range(1, 500):
    try:
        ob = exchange.fetch_order_book(symbol)

        # Convert order book to DataFrames
        bids_df = pd.DataFrame(ob["bids"], columns=["price", "amount"])
        asks_df = pd.DataFrame(ob["asks"], columns=["price", "amount"])

        if bids_df.empty or asks_df.empty:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Empty order book, skipping.")
            time.sleep(poll_interval)
            continue

        # Level 1 (Top of Book)
        best_bid = bids_df["price"].iloc[0]
        best_ask = asks_df["price"].iloc[0]
        bid_vol_1 = bids_df["amount"].iloc[0]
        ask_vol_1 = asks_df["amount"].iloc[0]

        # Core Pricing & Spread
        mid_price = (best_bid + best_ask) / 2
        spread = best_ask - best_bid
        fractional_price = mid_price - int(mid_price)

        # Micro-Price (Volume-Weighted Mid)
        micro_price = (best_bid * ask_vol_1 + best_ask * bid_vol_1) / (
            bid_vol_1 + ask_vol_1
        )

        # OBI - Level 1
        obi_1 = (bid_vol_1 - ask_vol_1) / (bid_vol_1 + ask_vol_1)

        # OBI - Depth 10
        top_bids_10 = bids_df.head(10)
        top_asks_10 = asks_df.head(10)
        bid_vol_10 = top_bids_10["amount"].sum()
        ask_vol_10 = top_asks_10["amount"].sum()
        depth_10 = bid_vol_10 + ask_vol_10
        obi_10 = (bid_vol_10 - ask_vol_10) / (bid_vol_10 + ask_vol_10)
        depth_imbalance_10 = (
            (bid_vol_10 - ask_vol_10) / depth_10 if depth_10 != 0 else 0.0
        )

        # Spoofing Indicator
        obi_diff = obi_10 - obi_1

        # --- SAVE TO CSV ---
        current_time = datetime.now()

        row_data = pd.DataFrame(
            [
                {
                    "timestamp": current_time,
                    "mid_price": mid_price,
                    "micro_price": micro_price,
                    "fractional_price": fractional_price,
                    "spread": spread,
                    "obi_1": obi_1,
                    "obi_10": obi_10,
                    "obi_diff": obi_diff,
                    "bid_vol_10": bid_vol_10,
                    "ask_vol_10": ask_vol_10,
                    "depth_10": depth_10,
                    "depth_imbalance_10": depth_imbalance_10,
                }
            ]
        )

        # Append to CSV, create header only if file does not exist
        header = not pd.io.common.file_exists(csv_file)
        row_data.to_csv(csv_file, mode="a", index=False, header=header)

        # Status line
        print(
            f"[{current_time.strftime('%H:%M:%S')}] "
            f"Mid: {mid_price:.2f} | OBI(10): {obi_10:+.2f} | Spread: {spread:.2f}"
        )

    except Exception as e:
        print(
            f"[{datetime.now().strftime('%H:%M:%S')}] API error ({type(e).__name__}). "
            "Skipping to next iteration."
        )

    time.sleep(poll_interval)

print("Data collection complete.")

Starting advanced data collection. Press Ctrl+C to stop.

[16:26:06] Mid: 74067.76 | OBI(10): -0.19 | Spread: 29.15
[16:26:11] Mid: 74070.02 | OBI(10): -0.23 | Spread: 25.94
[16:26:16] Mid: 74070.02 | OBI(10): -0.21 | Spread: 25.94
[16:26:21] Mid: 74066.68 | OBI(10): -0.21 | Spread: 32.29
[16:26:26] Mid: 74066.68 | OBI(10): -0.21 | Spread: 32.29
[16:26:31] Mid: 74066.68 | OBI(10): -0.21 | Spread: 32.29
[16:26:36] Mid: 74028.22 | OBI(10): -0.30 | Spread: 32.56
[16:26:41] Mid: 74028.22 | OBI(10): -0.30 | Spread: 32.56
[16:26:46] Mid: 74040.23 | OBI(10): -0.28 | Spread: 30.10
[16:26:51] Mid: 74040.23 | OBI(10): -0.26 | Spread: 30.10
[16:26:56] Mid: 74037.19 | OBI(10): -0.26 | Spread: 24.02
[16:27:01] Mid: 74014.20 | OBI(10): -0.27 | Spread: 28.25
[16:27:06] Mid: 74014.09 | OBI(10): -0.30 | Spread: 27.70
[16:27:11] Mid: 73992.55 | OBI(10): -0.30 | Spread: 31.05
[16:27:16] Mid: 73990.64 | OBI(10): -0.30 | Spread: 31.55
[16:27:21] Mid: 73974.65 | OBI(10): -0.29 | Spread: 63.29
[16:27:27] Mid

KeyboardInterrupt: 